Our package provides data access in a Python programming environment.

Here, we will start a Clustering analysis for the Pancreatic ductal adenocarcinoma (pdac).

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# from gpnotebook.tools.standard_imports import *
import os, re,sys
import yaml
import pandas as pd
import numpy as np


In [3]:
# project_dir = r"/Users/yingweihu/Documents/GitHub/glycoproteinnotebook-private/data/v1/projects/PDAC_P_PDC000271"
project_dir = r"/Users/yingweihu/Documents/GitHub/glycoproteinnotebook-private/data/v1/projects/LUAD_P_PDC000149"
data_dir = os.path.join(project_dir,"matrix")
meta_dir = os.path.join(project_dir,"meta")
job_dir = os.path.join(project_dir,"precomputed","cluster")
if not os.path.exists(job_dir):
    os.mkdir(job_dir)

In [4]:
data_path = os.path.join(data_dir, "DIG_nglycoform-peptide_matrix-abundances-MD_norm.tsv")
data_df = pd.read_csv(data_path,sep="\t", index_col = [0,1,2,3])
data_df

Intensity.Reference  \
Site                                               Gene    Sequence                                       Glycan                            
ENSP00000431932@33;ENSP00000385235@64              LARGE2  AAALDGDPGAGPGDHNRSDCGPQPPPPPK                  N4H6F0S1G0            13.068439   
                                                           AAALDGDPGAGPGDHNRSDCGPQPPPPPKCELLHVAIVCAGHNSSR N7H7F5S4G0            13.991116   
ENSP00000360616@233;ENSP00000360614@233;ENSP000... GRIN1   AAAMLNMTGSGYVWLVGER                            N3H6F2S1G0            13.809375   
ENSP00000358211@391;ENSP00000501491@362;ENSP000... HSPA12A AAAPDRTNPLNITLPFSFIDYYKK                       N8H6F0S0G0            14.193575   
ENSP00000405669@239;ENSP00000499208@239            KDM1B   AAATGNASPGK                                    N5H7F0S1G0            14.739654   
...                                                                                                                                   ...   
ENSP00000434586@21928;ENSP00000352154@22053;ENS... TTN     YTLTVENNSGSK                                   N3H5F1S1G0            16.044261   
ENSP00000256637@780;ENSP00000438597@643            SORT1   YVCGGRFLVHRYSVLQQHAEANGVDGVDALDTASHTNK         N7H9F1S0G0            12.973794   
ENSP00000261590@458                                DSG2    YVQNGTYTVK                                     N6H6F4S1G0            11.889773   
                                                                                                          N7H7F2S2G0            11.371782   
                                                                                                          N7H7F4S1G0            10.379389   

                                                                                                                      C3N-01799_T_01  \
Site                                               Gene    Sequence                                       Glycan                       
ENSP00000431932@33;ENSP00000385235@64              LARGE2  AAALDGDPGAGPGDHNRSDCGPQPPPPPK                  N4H6F0S1G0       13.560314   
                                                           AAALDGDPGAGPGDHNRSDCGPQPPPPPKCELLHVAIVCAGHNSSR N7H7F5S4G0       13.951974   
ENSP00000360616@233;ENSP00000360614@233;ENSP000... GRIN1   AAAMLNMTGSGYVWLVGER                            N3H6F2S1G0       13.810176   
ENSP00000358211@391;ENSP00000501491@362;ENSP000... HSPA12A AAAPDRTNPLNITLPFSFIDYYKK                       N8H6F0S0G0       14.016180   
ENSP00000405669@239;ENSP00000499208@239            KDM1B   AAATGNASPGK                                    N5H7F0S1G0       14.225727   
...                                                                                                                              ...   
ENSP00000434586@21928;ENSP00000352154@22053;ENS... TTN     YTLTVENNSGSK                                   N3H5F1S1G0             NaN   
ENSP00000256637@780;ENSP00000438597@643            SORT1   YVCGGRFLVHRYSVLQQHAEANGVDGVDALDTASHTNK         N7H9F1S0G0             NaN   
ENSP00000261590@458                                DSG2    YVQNGTYTVK                                     N6H6F4S1G0             NaN   
                                                                                                          N7H7F2S2G0             NaN   
                                                                                                          N7H7F4S1G0             NaN   

                                                                                                                      C3N-01799_N_01  \
Site                                               Gene    Sequence                                       Glycan                       
ENSP00000431932@33;ENSP00000385235@64              LARGE2  AAALDGDPGAGPGDHNRSDCGPQPPPPPK                  N4H6F0S1G0       12.704704   
                                                           AAALDGDPGAGPGDHNRSDCGPQPPPPPKCELLHVAIVCAGHNSSR N

In [5]:
meta_path= os.path.join(meta_dir, "LUAD_meta.txt")
meta_df = pd.read_csv(meta_path,sep="\t",header=[0,1])
meta_df

,case_id,Age,Sex,Tumor_Size_cm,Histologic_Grade,Tumor_necrosis,Path_Stage_pT,Path_Stage_pN,Stage,BMI,Tobacco_smoking_history,KEAP1_mutation,STK11_mutation,KRAS_mutation,SETD1B_mutation,TP53_mutation,EGFR_mutation
,data_type,CON,BIN,CON,ORD,BIN,ORD,ORD,ORD,CON,ORD,BIN,BIN,BIN,BIN,BIN,BIN
0,11LU013,59,Male,6.0,NaN,Not identified,pT3,pN1,Stage III,21.37,non-smoker,0.0,0.0,0.0,0.0,1.0,1.0
1,11LU016,62,Male,3.0,NaN,Not identified,pT1,pN2,Stage III,18.73,non-smoker,0.0,0.0,0.0,0.0,1.0,1.0
2,11LU022,53,Male,4.0,NaN,Present,pT2,pN0,Stage I,21.76,non-smoker,0.0,0.0,1.0,0.0,1.0,0.0
3,11LU035,59,Male,3.0,NaN,Not identified,pT1,pN0,Stage I,20.20,current smoker,0.0,0.0,0.0,0.0,1.0,0.0
4,C3L-00001,61,Female,3.7,G3 Poorly differentiated,Not identified,pT2,pN1,Stage II,22.03,non-smoker,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,C3N-02582,77,Male,5.8,G3 Poorly differentiated,Not identified,pT2,pN1,Stage II,23.30,past smoker,0.0,0.0,0.0,0.0,1.0,1.0
106,C3N-02586,73,Male,3.1,G2 Moderately differentiated,Not identified,pT2,pN1,Stage II,23.89,past smoker,0.0,0.0,0.0,0.0,1.0,0.0
107,C3N-02587,59,Female,2.0,G2 Moderately differentiated,Not identified,pT1,pN0,Stage I,21.93,non-smoker,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
meta_cols = ['case_id','Sex','Stage']
meta2 = meta_df.loc[:,meta_cols]
meta2.columns = ['Sample.ID'] + meta_cols[1:]
meta2

,Sample.ID,Sex,Stage
0,11LU013,Male,Stage III
1,11LU016,Male,Stage III
2,11LU022,Male,Stage I
3,11LU035,Male,Stage I
4,C3L-00001,Female,Stage II
...,...,...,...
105,C3N-02582,Male,Stage II
106,C3N-02586,Male,Stage II
107,C3N-02587,Female,Stage I
108,C3N-02588,Male,Stage II


In [7]:
meta2.head(17)

,Sample.ID,Sex,Stage
0,11LU013,Male,Stage III
1,11LU016,Male,Stage III
2,11LU022,Male,Stage I
3,11LU035,Male,Stage I
4,C3L-00001,Female,Stage II
5,C3L-00009,Male,Stage I
6,C3L-00080,Male,Stage I
7,C3L-00083,Male,Stage I
8,C3L-00093,Female,Stage I
9,C3L-00094,Male,Stage I


In [8]:
head_cols = ['Site', 'Gene', 'Sequence', 'Glycan', 'Intensity.Reference']
samples = [i for i in data_df.columns.values if i not in head_cols]
samples = [i for i in samples if i.split('_')[0] in list(meta2['Sample.ID']) and i.split('_')[1] == 'T']
len(samples)

112

In [9]:
rows = []
for sample in samples:
    key = sample.split('_')[0]
    row = meta2[meta2['Sample.ID']==key].iloc[0]
    row['Sample.ID'] = sample
    rows.append(row)
meta3 = pd.DataFrame(rows)

In [10]:
meta3.head(17)

,Sample.ID,Sex,Stage
83,C3N-01799_T_01,Male,Stage II
29,C3L-01890_T_01,Female,Stage I
58,C3N-00572_T_01,Female,Stage I
100,C3N-02423_T_01,Male,Stage I
109,C3N-02729_T_02,Male,Stage II
13,C3L-00263_T_02,Male,Stage II
76,C3N-01410_T_02,Male,Stage II
60,C3N-00578_T_02,Male,Stage III
107,C3N-02587_T_03,Female,Stage I
20,C3L-00893_T_03,Male,Stage I


In [11]:
meta3 = meta3.replace(np.nan,'NA')

In [12]:
top_ann_data_path = os.path.join(job_dir,'top_ann_data.tsv')
meta3.to_csv(top_ann_data_path, sep="\t", index=False)

Top annotation settings.

In [13]:

top_ann_settings = {
    'Sex': {
        'Male': 'blue',
        'Female': 'red',
        'NA': 'grey',
    },
    'Stage': {
        'Stage I': 'blue',
        'Stage II': 'green',
        'Stage III': 'orange',
        'Stage IV': 'red',
        'NA': 'grey'
    },

}
top_ann_settings_path = os.path.join(job_dir,'top_ann_settings.yml')
with open(top_ann_settings_path,'w') as f:
    yaml.dump(top_ann_settings,f,default_flow_style=False)

In [14]:
data_df.head(2)

Intensity.Reference  \
Site                                  Gene   Sequence                                       Glycan                            
ENSP00000431932@33;ENSP00000385235@64 LARGE2 AAALDGDPGAGPGDHNRSDCGPQPPPPPK                  N4H6F0S1G0            13.068439   
                                             AAALDGDPGAGPGDHNRSDCGPQPPPPPKCELLHVAIVCAGHNSSR N7H7F5S4G0            13.991116   

                                                                                                        C3N-01799_T_01  \
Site                                  Gene   Sequence                                       Glycan                       
ENSP00000431932@33;ENSP00000385235@64 LARGE2 AAALDGDPGAGPGDHNRSDCGPQPPPPPK                  N4H6F0S1G0       13.560314   
                                             AAALDGDPGAGPGDHNRSDCGPQPPPPPKCELLHVAIVCAGHNSSR N7H7F5S4G0       13.951974   

                                                                                                        C3N-01799_N_01  \
Site                                  Gene   Sequence                                       Glycan                       
ENSP00000431932@33;ENSP00000385235@64 LARGE2 AAALDGDPGAGPGDHNRSDCGPQPPPPPK                  N4H6F0S1G0       12.704704   
                                             AAALDGDPGAGPGDHNRSDCGPQPPPPPKCELLHVAIVCAGHNSSR N7H7F5S4G0       14.666861   

                                                                                                        C3L-01890_T_01  \
Site                                  Gene   Sequence                                       Glycan                       
ENSP00000431932@33;ENSP00000385235@64 LARGE2 AAALDGDPGAGPGDHNRSDCGPQPPPPPK                  N4H6F0S1G0       13.348172   
                                             AAALDGDPGAGPGDHNRSDCGPQPPPPPKCELLHVAIVCAGHNSSR N7H7F5S4G0       14.488220   

                                                                                                        C3L-01890_N_01  \
Site                                  Gene   Sequence                                       Glycan                       
ENSP00000431932@33;ENSP00000385235@64 LARGE2 AAALDGDPGAGPGDHNRSDCGPQPPPPPK                  N4H6F0S1G0       12.107846   
                                             AAALDGDPGAGPGDHNRSDCGPQPPPPPKCELLHVAIVCAGHNSSR N7H7F5S4G0       14.660851   

                                                                                                        C3N-00572_T_01  \
Site                                  Gene   Sequence                                       Glycan                       
ENSP00000431932@33;ENSP00000385235@64 LARGE2 AAALDGDPGAGPGDHNRSDCGPQPPPPPK                  N4H6F0S1G0       13.295248   
                                             AAALDGDPGAGPGDHNRSDCGPQPPPPPKCELLHVAIVCAGHNSSR N7H7F5S4G0       14.331695   

                                                                                                        C3N-00572_N_01  \
Site                                  Gene   Sequence                                       Glycan                       
ENSP00000431932@33;ENSP00000385235@64 LARGE2 AAALDGDPGAGPGDHNRSDCGPQPPPPPK                  N4H6F0S1G0       12.992695   
                                             AAALDGDPGAGPGDHNRSDCGPQPPPPPKCELLHVAIVCAGHNSSR N7H7F5S4G0       15.299847   

                                                                                                        C3N-02423_T_01  \
Site                                  Gene   Sequence                                       Glycan                       
ENSP00000431932@33;ENSP00000385235@64 LARGE2 AAALDGDPGAGPGDHNRSDCGPQPPPPPK                  N4H6F0S1G0       13.476404   
                                             AAALDGDPGAGPGDHNRSDCGPQPPPPPKCELLHVAIVCAGHNSSR N7H7F5S4G0       14.076331   

                                                                                                        C3N-02423_N_01  \
Site                                  Gene   Seque

In [15]:
data_df.shape

(102888, 251)

In [16]:
samples = meta3['Sample.ID'].to_list()

In [17]:
len(samples)

112

In [18]:
df2 = data_df.loc[:,samples].dropna()

In [19]:
df2.shape

(1345, 112)

In [20]:
from scipy.stats import variation
rows = []
for index,row in df2.iterrows():
    rows.append([variation([np.power(2,i) for i in list(row)])])
cv_df = pd.DataFrame(rows,columns=['cv'],index= df2.index)

glycopeptides = cv_df[cv_df['cv']>0.25].index

data2 = df2[df2.index.isin(glycopeptides)]
glycopeptides =  [f'{site}@{gene}@{seq}@{glycan}' for site,gene,seq,glycan in glycopeptides]
data2.index = glycopeptides
tumor_expression_path = os.path.join(job_dir,'expression_data.tsv')
data2.to_csv(tumor_expression_path,sep='\t',index=True)

In [21]:
data2.shape

(1093, 112)

Extract tumor samples from glycopeptide expression data based on pathological status,

calculates the coefficient of variation (CV) for each glycopeptide, selects glycopeptides with CV greater than 0.25.

Map glcopeptides with cv>0.25 in tumor patients with glycan type.

In [22]:
import re,os, sys

def decide_glycan_type(g):
    m = re.finditer("([A-Z])([\d]+)", g)
    y = [(i.group(1), int(i.group(2))) for i in m]
    d = dict(y)
    glycan_type = "Other"
    if d["N"] == 2 and d["H"] >= 5 and d["F"] == 0 and d["S"] == 0 and d["G"] == 0:
        glycan_type = "HM"
    elif d["N"] >= 2 and d["H"] >= 3 and d["F"] > 0 and d["S"] == 0:
        glycan_type = "only_F"
    elif d["N"] >= 2 and d["H"] >= 3 and d["S"] > 0 and d["F"] == 0:
        glycan_type = "only_S"
    elif d["N"] >= 2 and d["H"] >= 3 and d["S"] > 0 and d["F"] > 0:
        glycan_type = "F+S"
    return glycan_type


In [23]:
# left annotation
# from gpnotebook.tools.glycan import decide_glycan_type

glycan_type_map = dict(zip(glycopeptides,[decide_glycan_type(i) for i in glycopeptides]))
  
left_ann_data_path =  os.path.join(job_dir,'left_annotation_data.tsv')
rows = []
for i in glycan_type_map:
    rows.append([i,glycan_type_map[i]])
left_ann_data = pd.DataFrame(rows,columns=['Glycopeptide','GlycanType'])
left_ann_data.to_csv(left_ann_data_path,sep="\t",index=False)

In [24]:
left_ann_data

,Glycopeptide,GlycanType
0,ENSP00000418081@773;ENSP00000407393@773;ENSP00...,HM
1,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N3H...,F+S
2,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N4H...,F+S
3,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N5H...,F+S
4,ENSP00000273784@166;ENSP00000393887@165@AHSG@A...,only_S
...,...,...
1088,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S
1089,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S
1090,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S
1091,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S


Map glycan types with colors.

In [25]:

# left annotation settings, including color, order
left_ann_settings_path = os.path.join(job_dir,'left_annotation_settings.yml')
left_ann_settings = {
    "glycan_type_index" :{
    "HM": 1,
    "only_F":2,
    "only_S":3,
    "F+S":4,
    "Other":5
    },
    "glycan_type_color" : {
        "HM": 'green',
    "only_F": 'red',
    "only_S": 'purple',
    "F+S": 'orange',
    "Other": 'grey'
}
}
with open(left_ann_settings_path,'w') as f:
    yaml.dump(left_ann_settings,f,default_flow_style=False)
    

Parameters for NMF clustering.

In [26]:
nmf_parameters_path = os.path.join(job_dir, 'nmf_parameters.yml')
nmf_parameters = {
    'k_range': {
        'min': 3,
        'max': 5,
    },
    'test':{
        'nruns': 50
    },
    'opt_k':{
        'nruns': 500,
        'predefined': 0,
        'value': 4,
        'feature_prob': 0.8
    }
}
with open(nmf_parameters_path,'w') as f:
    yaml.dump(nmf_parameters,f,default_flow_style=False)

Generate a YAML configuration file (nmf_configs.yml) containing paths to various data required for NMF clustering.

In [27]:
config_data = {
    'input': {
        'expression_data': tumor_expression_path,
        'left_annotation_data': left_ann_data_path ,
        'left_annotation_settings': left_ann_settings_path,
        'top_annotation_data': top_ann_data_path,
        'top_annotatin_settings': top_ann_settings_path,
        'nmf_parameters': nmf_parameters_path
    },
    'output':{
        'out_dir': job_dir
    }
}
nmf_configs_path = os.path.join(job_dir,'nmf_configs.yml')
with open(nmf_configs_path,'w') as f:
    yaml.dump(config_data,f,default_flow_style=False)